In [ ]:
# ============================================================
# CREDORA FINANCE — Credit Risk & Loan Approval Analytics
# Notebook 04: Risk Scoring (0-100 score, scored_customers table, artefacts)
# PREREQUISITE: run 03_modeling.ipynb first in the same kernel session
# (needs: champion, champion_mode, scaler, THRESHOLD, X, y, results, sweep_df)
# ============================================================

In [ ]:
# Score ALL disbursed applications with the champion model.
proba_all = champion.predict_proba(
    scaler.transform(X) if champion_mode == 'scaled' else X)[:, 1]

scored = pd.DataFrame({
    'row_index': X.index,
    'default_probability': proba_all,
    'predicted_default': (proba_all >= THRESHOLD).astype(int),
    'risk_score': np.round(proba_all * 100).astype(int),
})
scored['risk_category'] = pd.cut(
    scored['risk_score'], bins=[-1, 33, 66, 100],
    labels=['Low', 'Medium', 'High'])

print(scored['risk_category'].value_counts())
print(f"\nMean risk score: {scored['risk_score'].mean():.1f}")

scored['risk_category'].value_counts().plot(
    kind='bar', color=['seagreen', 'darkorange', 'firebrick'],
    edgecolor='white')
plt.title('Customer Risk Distribution')
plt.ylabel('Applications')
plt.xticks(rotation=0)
plt.show()

In [ ]:
# FIXED: the original version concatenated scored rows onto disbursed_ids
# using row POSITION, trusting that a fresh SELECT * from MySQL comes
# back in exactly the same order as the disbursed subset built back in
# 02_feature_engineering. That is not guaranteed by MySQL and would
# silently attach a real customer's ID to the WRONG risk score with
# no error at all - the count-only assert can't catch it.
#
# Real fix: 02_feature_engineering must preserve application_id /
# customer_id as an index (not drop them, not one-hot them) so this
# cell can MERGE on that key instead of concatenating on position.
#
# If you haven't updated 02_feature_engineering to keep the ID as the
# DataFrame index yet, do that first - see the note in that notebook's
# Cell 5. This cell assumes X.index / model_df.index still holds the
# original application_id values.
url = URL.create(
    drivername="mysql+mysqlconnector",
    username="root",
    password=os.environ["MYSQL_PASSWORD"],
    host="localhost",
    database="credora_risk"
)
engine = create_engine(url)

loans = pd.read_sql("SELECT * FROM loan_applications", engine)
disbursed_keys = loans.loc[loans['disbursed_flag'] == 1,
                           ['application_id', 'customer_id']].copy()

# 'scored' must carry application_id (as its index, from X.index) —
# reset it into a real column so we can merge on a genuine key.
scored_keyed = scored.copy()
scored_keyed['application_id'] = scored_keyed['row_index'].map(
    dict(enumerate(X.index)))  # X.index must be application_id — verify this

scored_out = disbursed_keys.merge(
    scored_keyed.drop(columns='row_index'),
    on='application_id', how='inner', validate='one_to_one')

assert len(scored_out) == len(scored), \
    "Merge dropped or duplicated rows — check that X.index is really application_id"

scored_out.to_sql('scored_customers', engine,
                  if_exists='replace', index=False)
print(f"\nscored_customers written to MySQL: {len(scored_out):,} rows")

# Verify from the database side
check = pd.read_sql(
    "SELECT risk_category, COUNT(*) n FROM scored_customers "
    "GROUP BY risk_category", engine)
print(check)

In [ ]:
joblib.dump({'model': champion, 'scaler': scaler,
             'mode': champion_mode, 'threshold': THRESHOLD,
             'feature_names': list(X.columns)},
            'default_model.pkl')
results.round(4).to_csv('model_comparison.csv', index=False)
sweep_df.round(4).to_csv('threshold_sweep.csv', index=False)
scored_out.to_csv('scored_customers.csv', index=False)

print("""
WEEK 3 COMPLETE when:
[ ] 4 models compared with 5-fold CV ROC-AUC
[ ] Champion selected (expected: XGBoost or Random Forest)
[ ] ROC-AUC >= 0.80 and Recall >= 0.70 verified (or threshold tuned)
[ ] SHAP top drivers match domain sense (credit_score, DTI,
    utilization, prior defaults)
[ ] 0-100 risk score + Low/Medium/High category assigned
[ ] scored_customers table live in MySQL (feeds dashboard)
[ ] default_model.pkl + model card inputs saved

NEXT — Week 4: BI dashboard (8 PRD components), model card,
README, end-to-end reproducibility check, demo walkthrough.
""")